# 21 · Silver: ICA catastrophe events
**Owner: R** · Layer: silver · Runs after: 10, 20

**What:** turn `bronze.ica` into `silver.ica_events`: one typed, cleaned row per real event, **all years**, with flags that explain every decision.

**Why:** this is where the data-quality rules from profiling are applied. Silver keeps out-of-scope events (flagged `in_scope = false` with a reason). Gold decides what to load, so the scope can change without re-cleaning.

| Rule | Problem (profiling evidence) | Treatment |
|---|---|---|
| DQ1 | 27 dates such as `15-Oct-68` parse as 2068 | Pull future dates back 100 years; flag `date_century_fixed` |
| DQ1b | 3 events finish before they start | Flag `end_before_start`; duration NULL |
| DQ2 | 109 of 164 in-scope events have no state; others are regions or lists | Map via `xw_state`; first state is primary; infer blanks from keywords; else Unknown |
| DQ3 | Money and counts stored as text with `$`, `,`, `-` | Strip, cast; `-` = 0 (accounting nil); blank = NULL |
| DQ4 | 497 events share CAT code "Undeclared" | Business key is the source row: `event_id = 'ICA-<row>'` |
| DQ7 | Man-made and overseas events | `in_scope = false` with reason |
| DQ8 | Claim types don't sum to the total for 9 events | Keep total; flag `claims_parts_mismatch` |
| DQ15 | Kin Kin Tornado listed twice | Keep the more complete row; reject the other |
| DQ16 | State column omits states named in the event name | Flag `state_name_conflict` |

In [0]:
%run ./00_common

In [0]:
b = read("bronze", "ica")
rows_in = b.count()
CLAIM_COLS = ["domestic_building_claims", "domestic_content_claims", "domestic_motor_claims", "domestic_other_claims",
              "commercial_property_claims", "commercial_motor", "commercial_bi_claims", "commercial_other_claims",
              "commercial_crop_claims"]

s = b.select(
    "_row_num",
    F.concat(F.lit("ICA-"), F.lpad(F.col("_row_num").cast("string"), 4, "0")).alias("event_id"),          # DQ4
    F.upper(blank_to_null("cat_name")).alias("cat_code"),
    blank_to_null("event_name").alias("event_name"),
    blank_to_null("description").alias("description"),
    blank_to_null("town").alias("towns"), blank_to_null("postcode").alias("postcodes"),
    blank_to_null("type").alias("hazard_type_raw"),
    blank_to_null("state").alias("state_raw"),
    blank_to_null("financial_year").alias("ica_fy_label"),
    F.col("year").cast("int").alias("ica_year"),
    parse_date("event_start", "d-MMM-yy").alias("start_parsed"),
    parse_date("event_finish", "d-MMM-yy").alias("end_parsed"),
    num("original_loss_value", True).alias("original_loss_aud"),
    num("normalised_loss_value_2022", True).alias("normalised_loss_2022_aud"),
    num("total_claims_received", True).cast("int").alias("total_claims"),
    *[num(c, True).cast("int").alias(c if c.endswith("claims") else c + "_claims") for c in CLAIM_COLS],
    # how many source fields are filled: used to pick the better copy of a duplicate
    sum(F.when(F.trim(F.col(c)) != "", 1).otherwise(0) for c in [c for c in b.columns if not c.startswith("_")]).alias("_filled_fields"),
)
CLAIM_OUT = [c if c.endswith("claims") else c + "_claims" for c in CLAIM_COLS]

# DQ1: century fix, then derived date fields
s = (s.withColumn("start_date", fix_century(F.col("start_parsed")))
      .withColumn("end_date", fix_century(F.col("end_parsed")))
      .withColumn("date_century_fixed", F.col("start_date") != F.col("start_parsed"))
      .withColumn("end_before_start", F.col("end_date") < F.col("start_date"))
      .withColumn("duration_days", F.when(F.col("end_date") >= F.col("start_date"), F.datediff("end_date", "start_date") + 1))
      .withColumn("au_financial_year", F.when(F.month("start_date") >= 7, F.year("start_date") + 1).otherwise(F.year("start_date"))))

### Hard failures go to `audit.rejects`
Two rules remove rows from silver: an unparseable start date (an event with no date can't be placed anywhere) and a duplicate listing of the same event (DQ15).

In [0]:
bad_date = s.where(F.col("start_date").isNull())
n_bad_date = reject(bad_date, "silver", "ica", "DQ1", "Start date could not be parsed")
s = s.where(F.col("start_date").isNotNull())

w_dup = Window.partitionBy(F.lower("event_name"), "start_date").orderBy(F.col("_filled_fields").desc(), "_row_num")
s = s.withColumn("_dup_rank", F.row_number().over(w_dup))
dups = s.where("_dup_rank > 1")
display(dups.select("event_id", "event_name", "start_date", "state_raw", "_filled_fields"))
n_dup = reject(dups, "silver", "ica", "DQ15", "Duplicate of a more complete row with the same name and start date")
s = s.where("_dup_rank = 1").drop("_dup_rank")

### Hazard (crosswalk)

In [0]:
hz = read("silver", "xw_hazard").where("source = 'ica'").select(F.col("source_value").alias("hazard_type_raw"), "hazard_group", "hazard_family")
s = s.join(hz, "hazard_type_raw", "left")

### State (DQ2, DQ16)
1. **Reported:** split the State text on `,` `and` `&` `/`, map each part through `xw_state` (FNQ → QLD). The first part is the primary state.
2. **Inferred:** if nothing was reported, search the name, towns and description for state keywords. Accept only if exactly one state is found.
3. Otherwise **unknown**. The event stays, and gold points it to the Unknown location (key −1).

In [0]:
xw = read("silver", "xw_state")
parts = (s.select("event_id", F.posexplode(F.split("state_raw", r"(?i),|\band\b|&|/|;")).alias("pos", "part"))
          .withColumn("source_value", F.upper(F.trim("part")))
          .join(xw, "source_value", "left"))
reported = parts.groupBy("event_id").agg(
    F.min(F.when(F.col("state_code").isNotNull(), F.struct("pos", "state_code"))).getField("state_code").alias("reported_state"),
    F.array_sort(F.collect_set("state_code")).alias("reported_states"),
    F.count("*").alias("state_parts"))

kw = read("silver", "xw_state_keyword")
def keyword_states(text_col, alias):
    t = s.select("event_id", F.lower(text_col).alias("_t"))
    hits = t.crossJoin(kw).where(F.expr(r"_t rlike concat('\\b', keyword, '\\b')"))
    return hits.groupBy("event_id").agg(F.array_sort(F.collect_set("state_code")).alias(alias))

inferred = keyword_states(F.concat_ws(" ", "event_name", "towns", "description"), "text_states")
named = keyword_states(F.col("event_name"), "name_states")

s = (s.join(reported, "event_id", "left").join(inferred, "event_id", "left").join(named, "event_id", "left")
      .withColumn("outside_australia", F.coalesce(F.lower("state_raw").contains("outside"), F.lit(False)))
      .withColumn("state_code", F.when(F.col("outside_australia"), None)
                  .when(F.col("reported_state").isNotNull(), F.col("reported_state"))
                  .when(F.size("text_states") == 1, F.col("text_states")[0]))
      .withColumn("state_source", F.when(F.col("outside_australia"), "outside")
                  .when(F.col("reported_state").isNotNull(), "reported")
                  .when(F.size("text_states") == 1, "inferred")
                  .otherwise("unknown"))
      .withColumn("multi_state_flag", (F.coalesce(F.col("state_parts"), F.lit(0)) > 1) | (F.size(F.coalesce("name_states", F.array())) > 1))
      .withColumn("state_name_conflict", F.col("reported_state").isNotNull() & (F.size(F.coalesce("name_states", F.array())) > 0)
                  & (F.size(F.array_except(F.coalesce("name_states", F.array()), F.coalesce("reported_states", F.array()))) > 0))
      .withColumn("all_states", F.when(F.col("state_source") == "reported", F.array_join("reported_states", ","))
                  .otherwise(F.array_join(F.coalesce("text_states", F.array()), ","))))

### Claims consistency (DQ8) and scope (DQ7)

In [0]:
parts_sum = sum(F.coalesce(F.col(c), F.lit(0)) for c in CLAIM_OUT)
s = (s.withColumn("claims_parts_mismatch", F.col("total_claims").isNotNull() & (parts_sum != F.col("total_claims")))
      .withColumn("declaration_status", F.when(F.col("cat_code") == "UNDECLARED", "Undeclared").otherwise("Declared"))
      .withColumn("exclusion_reason", F.when(F.col("start_date") < F.lit(SCOPE_START).cast("date"), "before_scope_start")
                  .when(F.col("outside_australia"), "outside_australia")
                  .when(F.col("hazard_group").isNull(), "non_natural_hazard"))
      .withColumn("in_scope", F.col("exclusion_reason").isNull()))

cols = ["event_id", "_row_num", "cat_code", "declaration_status", "event_name", "hazard_type_raw", "hazard_group", "hazard_family",
        "start_date", "end_date", "duration_days", "date_century_fixed", "end_before_start", "ica_year", "ica_fy_label", "au_financial_year",
        "state_raw", "state_code", "state_source", "all_states", "multi_state_flag", "state_name_conflict",
        "towns", "postcodes", "description", "original_loss_aud", "normalised_loss_2022_aud", "total_claims", *CLAIM_OUT,
        "claims_parts_mismatch", "in_scope", "exclusion_reason"]
n_out = write(s.select(*cols).withColumnRenamed("_row_num", "source_row"), "silver", "ica_events")
log_step("silver", "clean_ica", "silver.ica_events", rows_in, n_out, n_bad_date + n_dup)

### Results (the "after" side of the data-quality log)

In [0]:
ev = read("silver", "ica_events")
display(ev.groupBy("in_scope", "exclusion_reason").count().orderBy("in_scope"))
display(ev.where("in_scope").groupBy("state_source").count().orderBy("state_source"))
display(ev.agg(F.sum(F.col("date_century_fixed").cast("int")).alias("dates_fixed_DQ1"),
               F.sum(F.col("end_before_start").cast("int")).alias("end_before_start_DQ1b"),
               F.sum(F.col("claims_parts_mismatch").cast("int")).alias("claims_mismatch_DQ8"),
               F.sum(F.col("state_name_conflict").cast("int")).alias("state_name_conflict_DQ16"),
               F.sum(F.when(F.year("start_date") != F.col("ica_year"), 1).otherwise(0)).alias("year_mismatch_after_fix")))